In [51]:
import pandas as pd
import numpy as np

In [52]:
data = pd.read_csv("AmesHousing.csv")

In [53]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   str    
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot Shape        2930 non-null   str    
 9   Land Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot Config       2930 non-null   str    
 12  Land Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition 1      2930 non-null   str    
 15  Condition 2      2930 non-null   str    
 16  Bldg Type        2930 non-null   str    
 17  House Style      2930 non

In [54]:
data["SalePrice"].min()

np.int64(12789)

In [55]:
data["SalePrice"].max()

np.int64(755000)

In [56]:
data = data.drop(columns=["Order", "PID"])

In [57]:
test_data = data.sample(frac=0.2, random_state=42) 
data = data.drop(test_data.index)
valid_data = data.sample(frac=0.2, random_state=42)
train_data = data.drop(valid_data.index)

### Питання: Чому дані розбиваються ДО масштабування/кодування, а не після?

Якщо параметри перетворень (середнє і std для стандартизації, список категорій для
One-Hot Encoding, статистики для Target Encoding тощо) обчислювати на **всьому** датасеті
(до розбиття), то ці параметри неявно "побачать" інформацію з Validation/Test вибірок.
Модель тоді опосередковано отримує доступ до розподілу даних, яких у реальному житті
на момент навчання ще не існує (майбутні спостереження) — це і називається **Data Leakage**
(витік даних).

Наслідки:
- Метрики на Validation/Test виявляються завищеними (оптимістичними), бо вибірки вже
  "узгоджені" з train через спільні статистики.
- Оцінка якості моделі стає ненадійною: у продакшені на справді нових даних якість
  виявиться гіршою, ніж показували тестові метрики.

Тому правильний порядок: спочатку **split** на Train/Validation/Test, а вже потім усі
статистики (mean/std для StandardScaler, категорії для OHE, mode/медіана для імпутації)
рахуються **лише на Train**, і застосовуються (`transform`, без повторного `fit`) до
Validation і Test.

In [58]:
train_data.columns

Index(['MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street',
       'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu',
       'Garage Type', 'Garage Yr Blt', 'Garage Finish', 'Gara

Даних занадто багато

In [59]:
missing_counts = train_data.isna().sum()

columns_with_na = missing_counts[missing_counts > 0]

print(columns_with_na)

Lot Frontage       312
Alley             1747
Mas Vnr Type      1150
Mas Vnr Area        16
Bsmt Qual           50
Bsmt Cond           50
Bsmt Exposure       50
BsmtFin Type 1      50
BsmtFin SF 1         1
BsmtFin Type 2      51
BsmtFin SF 2         1
Bsmt Unf SF          1
Total Bsmt SF        1
Bsmt Full Bath       1
Bsmt Half Bath       1
Fireplace Qu       905
Garage Type         95
Garage Yr Blt       97
Garage Finish       97
Garage Cars          1
Garage Area          1
Garage Qual         97
Garage Cond         97
Pool QC           1864
Fence             1499
Misc Feature      1801
dtype: int64


У деяких колонках є забагато пропусків, проте це не означає, що вони невалідні: це просто може означати про відсутність певної компоненти у будинку. Для цього кожну колонку варто розглядати окремо

In [60]:
import pandas as pd

def generate_missing_values_html(df, filename="missing_values_report.html", max_vals_display=20):
    report_data = []
    cols_with_na = df.columns[df.isna().any()].tolist()
    
    for col in cols_with_na:
        missing_count = df[col].isna().sum()
        
        unique_vals = df[col].dropna().unique()
        n_unique = len(unique_vals)
        
        if n_unique > max_vals_display:
            display_str = ", ".join(map(str, unique_vals[:max_vals_display])) + f" ... (та ще {n_unique - max_vals_display})"
        else:
            display_str = ", ".join(map(str, unique_vals))
            
        report_data.append({
            "Колонка": col,
            "Кількість пропусків": missing_count,
            "Тип даних": df[col].dtype,
            "Кількість унікальних": n_unique,
            "Значення": display_str
        })

    if not report_data:
        print("Пропусків у датасеті не знайдено.")
        return
        
    report_df = pd.DataFrame(report_data)
    report_df = report_df.sort_values(by="Кількість пропусків", ascending=False)
    
    html_content = f"""
    <!DOCTYPE html>
    <html lang="uk">
    <head>
        <meta charset="UTF-8">
        <title>Звіт по колонках із пропусками</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; }}
            h2 {{ color: #333; }}
            table {{ border-collapse: collapse; width: 100%; }}
            th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
            th {{ background-color: #d9534f; color: white; }}
            tr:nth-child(even) {{ background-color: #f9f9f9; }}
            tr:hover {{ background-color: #f1f1f1; }}
        </style>
    </head>
    <body>
        <h2>Звіт по колонках із пропусками (NaN)</h2>
        {report_df.to_html(index=False, border=0)}
    </body>
    </html>
    """
    
    with open(filename, "w", encoding="utf-8") as file:
        file.write(html_content)
        
    print(f"Звіт успішно збережено у файл: {filename}")


generate_missing_values_html(data)

Звіт успішно збережено у файл: missing_values_report.html


За генерованим звітом можна побачити, що майже всі категоріальні колонки не мають міток відсутності чого-небудь, тобто None несе тут роль відсутності. Це застосовно до наступних колонок: 
Bsmt Qual, Bsmt Cond, Bsmt Exposure, BsmtFin Type 1, BsmtFin Type 2, Garage Type, Garage Finish, Garage Qual, Garage Cond, Pool QC, Misc Feature, Alley, Fence, Mas Vnr Type, Fireplace Qu, Electrician.
Такі колонки як Bsmt Exposure, BsmtFin Type 2, Garage_Type мають None як пропуск.
Числові ознаки типу Mas Vnr Area (23), Garage Area (1), Garage Cars (1), BsmtFin SF 1 (1), BsmtFin SF 2 (1), Bsmt Unf SF (1), Total Bsmt SF (1), Bsmt Full Bath (2), Bsmt Half Bath заповнюємо 0 так як вони доповнюють концепцію відсутності чогось
Lot Frontage - пропуск
з garage yrs build ще не вибудувалася стратегія(надалі задля усунення мультиколінеарності колонка буде видалена)

In [61]:
null_cat = ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
      "BsmtFin Type 1", "Garage Type",
      "Garage Qual", "Garage Cond", "Pool QC",
      "Misc Feature", "Alley", "Fence", "Mas Vnr Type", "Fireplace Qu",
        "Electrical"]
null_cat_miss = ["Bsmt Exposure", "BsmtFin Type 2", "Garage Finish"]
null_num = ["Mas Vnr Area", "Garage Area", "Garage Cars", "BsmtFin SF 1",
             "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF",
               "Bsmt Full Bath", "Bsmt Half Bath"]

In [62]:
train_data[null_cat] = train_data[null_cat].fillna("Empty")
train_data[null_num] = train_data[null_num].fillna(0)
cat_miss_mode = train_data[null_cat_miss].mode().iloc[0]
train_data[null_cat_miss] = train_data[null_cat_miss].fillna(cat_miss_mode)
lot_front_mode = train_data["Lot Frontage"].mode().iloc[0]
train_data["Lot Frontage"] = train_data["Lot Frontage"].fillna(lot_front_mode)

test_data[null_cat] = test_data[null_cat].fillna("Empty")
test_data[null_num] = test_data[null_num].fillna(0)
test_data[null_cat_miss] = test_data[null_cat_miss].fillna(cat_miss_mode)
test_data["Lot Frontage"] = test_data["Lot Frontage"].fillna(lot_front_mode)

valid_data[null_cat] = valid_data[null_cat].fillna("Empty")
valid_data[null_num] = valid_data[null_num].fillna(0)
valid_data[null_cat_miss] = valid_data[null_cat_miss].fillna(cat_miss_mode)
valid_data["Lot Frontage"] = valid_data["Lot Frontage"].fillna(lot_front_mode)

In [63]:
na_status = train_data.isna().any()
print(na_status[na_status])

Garage Yr Blt    True
dtype: bool


In [64]:
import seaborn as sns
import matplotlib.pyplot as plt

In [65]:
numeric_data = train_data.select_dtypes(include=['number'])

corr_matrix = numeric_data.corr()
target_corr = corr_matrix['SalePrice'].abs().sort_values(ascending=False)

print("Ознаки, що сильно або помірно корелюють із SalePrice:")
print(target_corr[target_corr > 0.4]) 
print("-" * 40)

# Мультиколінеарність рахуємо ТІЛЬКИ між ознаками, без SalePrice,
# бо кореляція ознаки з ціллю - це не мультиколінеарність, а бажана властивість ознаки
feature_corr_matrix = numeric_data.drop(columns="SalePrice").corr()
corr_pairs = feature_corr_matrix.unstack()

corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) < corr_pairs.index.get_level_values(1)]

multicollinearity = corr_pairs[abs(corr_pairs) > 0.75].sort_values(ascending=False)
print("Мультиколінеарність (пари ознака-ознака з |r| > 0.75):")
print(multicollinearity)
print("-" * 40)


Ознаки, що сильно або помірно корелюють із SalePrice:
SalePrice         1.000000
Overall Qual      0.791676
Gr Liv Area       0.702624
Garage Cars       0.642911
Garage Area       0.639118
Total Bsmt SF     0.616216
1st Flr SF        0.613992
Full Bath         0.544774
Year Built        0.541970
Year Remod/Add    0.520443
Garage Yr Blt     0.515387
Mas Vnr Area      0.490409
TotRms AbvGrd     0.474598
Fireplaces        0.462962
BsmtFin SF 1      0.424471
Name: SalePrice, dtype: float64
----------------------------------------
Мультиколінеарність (пари ознака-ознака з |r| > 0.75):
Garage Area    Garage Cars      0.888071
Garage Yr Blt  Year Built       0.817017
1st Flr SF     Total Bsmt SF    0.813192
Gr Liv Area    TotRms AbvGrd    0.806177
dtype: float64
----------------------------------------


In [66]:
na_status = train_data.isna().any()
print(na_status[na_status])

Garage Yr Blt    True
dtype: bool


У даних є роки, тож є можливість зменшити розмірність даних за рахунок знаходженням різниці між роком побудови/ренновації та продажу

In [67]:
train_data["year_build_sold"] = train_data["Yr Sold"] - train_data["Year Built"]
train_data["year_remod_sold"] = train_data["Yr Sold"] - train_data["Year Remod/Add"]

cols_to_compare = [
    "Year Built", "Yr Sold", "Year Remod/Add", 
    "year_build_sold", "year_remod_sold", 
    "SalePrice"
]

# Окремі назви змінних (НЕ corr_matrix/target_corr!), щоб не затерти
# "глобальну" матрицю кореляцій з клітинки 18 - вона ще знадобиться
# далі (клітинка 27) для відбору ознак за мультиколінеарністю
year_analysis_corr_matrix = train_data[cols_to_compare].corr()
year_target_corr = year_analysis_corr_matrix['SalePrice'].sort_values(ascending=False)

print(year_target_corr)


SalePrice          1.000000
Year Built         0.541970
Year Remod/Add     0.520443
Yr Sold           -0.034530
year_remod_sold   -0.522926
year_build_sold   -0.542349
Name: SalePrice, dtype: float64


In [68]:
na_status = train_data.isna().any()
print(na_status[na_status])

Garage Yr Blt    True
dtype: bool


In [69]:
train_data = train_data.drop(columns=["Yr Sold", "Year Remod/Add", "Year Built"])

test_data["year_build_sold"] = test_data["Yr Sold"] - test_data["Year Built"]
test_data["year_remod_sold"] = test_data["Yr Sold"] - test_data["Year Remod/Add"]
test_data = test_data.drop(columns=["Yr Sold", "Year Remod/Add", "Year Built"])

valid_data["year_build_sold"] = valid_data["Yr Sold"] - valid_data["Year Built"]
valid_data["year_remod_sold"] = valid_data["Yr Sold"] - valid_data["Year Remod/Add"]
valid_data = valid_data.drop(columns=["Yr Sold", "Year Remod/Add", "Year Built"])

Трансформація абсолютних часових показників (Year Built, Year Remod/Add, Yr Sold) у відносні метрики віку об'єкта на момент продажу (year_build_sold, year_remod_sold) дозволила підвищити прогностичну силу ознак, оскільки нові змінні продемонстрували сильнішу кореляцію з цільовою змінною SalePrice. Ця агрегація допомогла зменшити розмірність датасету, замінивши три початкові колонки на дві більш змістовні, та успішно усунула шум від колонки Yr Sold, яка сама по собі мала вкрай слабкий зв'язок із ціною. Перехід від абсолютних дат до безперервних величин віку покращує математичну логіку даних, роблячи їх значно ефективнішими для обробки алгоритмами машинного навчання без втрати корисної інформації.

In [70]:
from scipy.stats import f_oneway, ks_2samp
from scipy.stats import f_oneway

def assess_cat_cols_anova(data, col):
    valid_data = data[[col, 'SalePrice']]
    unq_classes = valid_data[col].unique()
    if len(unq_classes) < 2:
        return 0
    grouped_data = [valid_data['SalePrice'][valid_data[col] == cat] for cat in unq_classes]
    f_stat, p_value = f_oneway(*grouped_data)
    if p_value > 0.05: 
        f_stat = 0
        
    return f_stat

def assess_cat_cols_ks2(data, col, target='SalePrice', alpha=0.05):
    """
    Оцінює значущість категоріальної ознаки за допомогою 2-вибіркового 
    критерію Колмогорова-Смірнова (One-vs-Rest).
    Повертає True, якщо колонка є статистично значущою для цільової змінної.
    """
    clean_data = data[[col, target]].dropna()
    categories = clean_data[col].unique()

    if len(categories) < 2:
        return False, 1.0
        
    min_p_value = 1.0
    
    for cat in categories:
        group_in = clean_data[clean_data[col] == cat][target]
        group_out = clean_data[clean_data[col] != cat][target]
        
        if len(group_in) == 0 or len(group_out) == 0:
            continue

        stat, p_value = ks_2samp(group_in, group_out)
        
        if p_value < min_p_value:
            min_p_value = p_value
            
    is_significant = min_p_value < alpha
    
    return is_significant, min_p_value
    


In [71]:
cat_cols = data.select_dtypes(exclude=['number']).columns
cols_to_drop = []

for col in cat_cols:

    anova_stat = assess_cat_cols_anova(data, col)
    is_ks_significant, ks_p_value = assess_cat_cols_ks2(data, col)
    

    if anova_stat == 0 and not is_ks_significant:
        cols_to_drop.append(col)

print(f"Кількість категоріальних колонок на видалення: {len(cols_to_drop)}")
print(f"Список на видалення: {cols_to_drop}")

train_data = train_data.drop(columns=cols_to_drop)
test_data = test_data.drop(columns=cols_to_drop)
valid_data = valid_data.drop(columns=cols_to_drop)

C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_oneway(*grouped_data)
C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_oneway(*grouped_data)
C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_oneway(*grouped_data)
C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_on

Кількість категоріальних колонок на видалення: 0
Список на видалення: []


C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_oneway(*grouped_data)
C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_oneway(*grouped_data)
C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_oneway(*grouped_data)
C:\Users\vital\AppData\Local\Temp\ipykernel_9868\1036491984.py:10: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_stat, p_value = f_on

In [72]:
# Перераховуємо кореляції на АКТУАЛЬНОМУ train_data - тобто вже ПІСЛЯ інженерії
# ознак з роками (клітинки 21-23) та видалення незначущих категоріальних колонок
# (клітинка 26). Раніше тут помилково використовувались застарілі corr_matrix /
# corr_pairs / target_corr з клітинок 18 та 21, через що майже всі ознаки мали
# target_corr == 0, і рішення про видалення були фактично випадковими, а сам
# таргет SalePrice потрапляв у пари "мультиколінеарності".

numeric_data = train_data.select_dtypes(include=['number'])

# Кореляція кожної числової ознаки з ціллю (SalePrice сюди не входить)
target_corr = numeric_data.corr()['SalePrice'].abs().drop('SalePrice').sort_values(ascending=False)

# Матриця кореляцій ознака-ознака БЕЗ SalePrice, щоб пара "ознака-таргет"
# не потрапляла у список мультиколінеарних (це різні речі!)
feature_corr_matrix = numeric_data.drop(columns='SalePrice').corr()
corr_pairs = feature_corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) < corr_pairs.index.get_level_values(1)]

threshold_drop = 0.75
high_corr_pairs = corr_pairs[abs(corr_pairs) > threshold_drop].sort_values(ascending=False)

print(f"Мультиколінеарність (пари ознака-ознака з |r| > {threshold_drop}) на актуальних даних:")
print(high_corr_pairs)
print("-" * 40)

features_to_drop = set()

for (feat1, feat2), corr_value in high_corr_pairs.items():
    if feat1 in features_to_drop or feat2 in features_to_drop:
        continue

    corr1_target = target_corr.get(feat1, 0)
    corr2_target = target_corr.get(feat2, 0)

    if corr1_target < corr2_target:
        features_to_drop.add(feat1)
    else:
        features_to_drop.add(feat2)

print(f"Ознаки, що видаляються через сильну мультиколінеарність (|r| > {threshold_drop}):")
print(features_to_drop)

train_data = train_data.drop(columns=list(features_to_drop))
test_data = test_data.drop(columns=list(features_to_drop))
valid_data = valid_data.drop(columns=list(features_to_drop))


Мультиколінеарність (пари ознака-ознака з |r| > 0.75) на актуальних даних:
Garage Area    Garage Cars        0.888071
1st Flr SF     Total Bsmt SF      0.813192
Gr Liv Area    TotRms AbvGrd      0.806177
Garage Yr Blt  year_build_sold   -0.815889
dtype: float64
----------------------------------------
Ознаки, що видаляються через сильну мультиколінеарність (|r| > 0.75):
{'Garage Yr Blt', 'Garage Area', 'TotRms AbvGrd', '1st Flr SF'}


In [73]:
low_corr_threshold = 0.5
numeric_data = train_data.select_dtypes(include=['number'])
corr_matrix = numeric_data.corr()

target_corr = corr_matrix['SalePrice'].abs().sort_values(ascending=False)
weak_features = target_corr[target_corr < low_corr_threshold].index.tolist()

print(f"Кількість слабких ознак на видалення (|r| < {low_corr_threshold}): {len(weak_features)}")
print(f"Список: {weak_features}")

valid_data = valid_data.drop(columns=weak_features)
test_data = test_data.drop(columns=weak_features)
train_data = train_data.drop(columns=weak_features)

Кількість слабких ознак на видалення (|r| < 0.5): 24
Список: ['Mas Vnr Area', 'Fireplaces', 'BsmtFin SF 1', 'Wood Deck SF', 'Lot Frontage', 'Open Porch SF', 'Bsmt Full Bath', 'Half Bath', '2nd Flr SF', 'Lot Area', 'Bsmt Unf SF', 'Bedroom AbvGr', 'Screen Porch', 'Enclosed Porch', 'Kitchen AbvGr', 'Overall Cond', 'Pool Area', 'MS SubClass', '3Ssn Porch', 'BsmtFin SF 2', 'Mo Sold', 'Low Qual Fin SF', 'Bsmt Half Bath', 'Misc Val']


In [74]:
train_data.to_csv("train_data.csv", index= False)
test_data.to_csv("test_data.csv", index= False)
valid_data.to_csv("valid_data.csv", index= False)

In [75]:
na_status = train_data.isna().any()
print(na_status[na_status])

Series([], dtype: bool)
